In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (confusion_matrix, classification_report,
                             roc_curve, roc_auc_score, RocCurveDisplay)
import matplotlib.pyplot as plt
import SurvivalToDischarge
import DecompensationImputed

C:\Users\redja\AppData\Local\Temp\ipykernel_15004\665133681.py:2: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


## Data

In [6]:
X, y = SurvivalToDischarge.load_data(sample_size=40000, icu_vitals=False, top_n_labs=0, top_n_drugs=200, top_n_procedures=200)

Returing 40000 patient records with 369 columns. y distribution: 
hospital_expire_flag
0    0.98005
1    0.01995
Name: proportion, dtype: float64


In [2]:
X, y = DecompensationImputed.load_data(sample_size=40000, icu_vitals=False, top_n_labs=0, top_n_drugs=200, top_n_procedures=200)

Returing 40000 patient records with 368 columns. y distribution: 
0    0.971025
1    0.028975
Name: proportion, dtype: float64


In [ ]:
# 80/20 train-test, need to switch to train-validate-test once were comparing models
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Lasso is scale-sensitive
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"Train: {X_train_scaled.shape}, Test: {X_test_scaled.shape}")

## Logistic Regression

In [ ]:
model = LogisticRegression(C=0.2, max_iter=10000)
model.fit(X_train_scaled, y_train)

In [ ]:
y_pred  = model.predict(X_test_scaled)
y_proba = model.predict_proba(X_test_scaled)[:, 1]

# --- Confusion matrix ---
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()
tpr = tp / (tp + fn)   # sensitivity / recall
fpr = fp / (fp + tn)   # fall-out
tnr = tn / (tn + fp)   # specificity
ppv = tp / (tp + fp)   # precision

print("Confusion matrix:")
print(f"  {'':12s}  Pred 0   Pred 1")
print(f"  {'Actual 0':12s}  {tn:>6}   {fp:>6}   (specificity {tnr:.3f})")
print(f"  {'Actual 1':12s}  {fn:>6}   {tp:>6}   (sensitivity {tpr:.3f})")
print()
print(f"TPR (sensitivity):  {tpr:.3f}")
print(f"FPR (fall-out):     {fpr:.3f}")
print(f"Precision:          {ppv:.3f}")
print(f"ROC-AUC:            {roc_auc_score(y_test, y_proba):.3f}")
print()
print(classification_report(y_test, y_pred, target_names=['Survived', 'Died']))

# --- Plots ---
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Confusion matrix heatmap
im = axes[0].imshow(cm, cmap='Blues')
axes[0].set_xticks([0, 1]); axes[0].set_yticks([0, 1])
axes[0].set_xticklabels(['Pred 0', 'Pred 1'])
axes[0].set_yticklabels(['Actual 0', 'Actual 1'])
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, cm[i, j], ha='center', va='center',
                     color='white' if cm[i, j] > cm.max() / 2 else 'black', fontsize=14)
axes[0].set_title('Confusion Matrix')
plt.colorbar(im, ax=axes[0])

# ROC curve
fpr_curve, tpr_curve, _ = roc_curve(y_test, y_proba)
auc = roc_auc_score(y_test, y_proba)
axes[1].plot(fpr_curve, tpr_curve, lw=2, label=f'AUC = {auc:.3f}')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1)
axes[1].scatter([fpr], [tpr], color='red', zorder=5, label=f'Operating point\nFPR={fpr:.2f}, TPR={tpr:.2f}')
axes[1].set_xlabel('False Positive Rate'); axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve'); axes[1].legend(fontsize=8)

# Top feature coefficients
coef_series = pd.Series(model.coef_[0], index=X.columns)
top_coef = coef_series[coef_series != 0].abs().sort_values(ascending=False).head(20)
signed_top = coef_series[top_coef.index]
colors = ['steelblue' if v > 0 else 'tomato' for v in signed_top]
axes[2].barh(signed_top.index[::-1], signed_top.values[::-1], color=colors[::-1])
axes[2].axvline(0, color='black', linewidth=0.8)
axes[2].set_xlabel('Logistic Regression Coefficient')
axes[2].set_title('Top 20 Features by |Coefficient|')
axes[2].tick_params(axis='y', labelsize=7)

plt.tight_layout()
plt.savefig('results/lasso_results.png', dpi=120, bbox_inches='tight')
plt.show()